## Question

Can we prepare a balanced dataset that teaches a model to route music-service messages to the correct part of the application?

Data preparation is the foundation of fine-tuning. A model can only learn the categories, boundaries, and mistakes represented in its training examples. Duplicate leakage, inconsistent labels, weak category definitions, or unrepresentative messages can make evaluation misleading and produce an unreliable classifier. For that reason, data preparation has its own notebook, where the dataset can be inspected and validated before any training begins.

The classifier needs to distinguish six intents: recommendation requests, recommendation-explanation requests, customer support, partnerships, junk, and messages that need human review. An explanation request can be sent to the RAG workflow built in the previous notebook, while the other categories follow their own routes.

## Hypothesis

If we define one source file per intent, remove exact duplicates before splitting, and split every intent separately, the resulting training, validation, and test sets should contain all six categories without exact-message leakage between splits.

Formatting the examples with Qwen's chat template should then produce training records containing the expected answer and test records that keep the answer separate for evaluation.

## Environment Setup

Create or activate the project environment and install the Python dependencies before running the notebook:

```sh
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

Select the `.venv` kernel in Jupyter. Install Ollama separately, then pull the corresponding quantized Ollama build of the instruction-tuned Qwen model used by the classification experiment:

```sh
ollama pull qwen2.5:0.5b-instruct
```

The Ollama tag is a Q4_K_M build of the same model family; it is not numerically identical to the Hugging Face checkpoint. Ollama will run the prompting baseline and, after fine-tuning, the imported model. This preparation notebook still uses `AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")` to produce Qwen-compatible training text. The first tokenizer load may download files from Hugging Face unless they are already cached.

Ollama runs models but does not perform the fine-tuning step. A training framework will create the fine-tuned weights or adapter, which can then be imported into Ollama. This notebook only prepares and formats data, so it does not require model-training hardware.

## Experiment

We will load twelve example messages for each intent, normalize them for duplicate detection, and assign numeric labels from the order of `source_files`. Each intent is shuffled with a fixed random seed and split approximately 80%/10%/10%.

After saving the labelled data, we will format the training and validation examples as complete conversations. Test prompts omit the assistant answer so a later notebook can compare the model's prediction with the stored label.

### Build the labelled dataset

The first code cell performs the main data-quality work. It accepts execution from either the repository root or the notebook directory, checks every CSV for a `text` column and non-empty messages, and rejects identical messages assigned to conflicting labels.

`casefold()` and whitespace normalization make duplicate comparison insensitive to capitalization and repeated spaces. The original message text is preserved in the saved dataset. `ClassLabel` stores the category names with the numeric labels so later notebooks do not have to reproduce the list order. `utf-8-sig` accepts ordinary UTF-8 CSVs as well as files containing a byte-order mark.

In [ ]:
import csv
import random
from pathlib import Path

from datasets import ClassLabel, Dataset, DatasetDict, Features, Value, load_from_disk

notebook_dir = Path.cwd()
if not (notebook_dir / "intentData").is_dir():
    notebook_dir = notebook_dir / "embeddings-and-search"

data_dir = notebook_dir / "data"
label_names = [
    "Recommendation_Request",
    "Explanation_Request",
    "Customer_Support",
    "Partnership",
    "Junk",
    "Human_Review",
]
source_files = [
    "recommendation_request.csv",
    "explanation_request.csv",
    "customer_support.csv",
    "partnership.csv",
    "junk.csv",
    "human_review.csv",
]

rng = random.Random(42)
split_records = {"train": [], "validation": [], "test": []}
seen = {}

for label, filename in enumerate(source_files):
    records = []
    with (notebook_dir / "intentData" / filename).open(
        newline="", encoding="utf-8-sig"
    ) as file:
        reader = csv.DictReader(file)
        if not reader.fieldnames or "text" not in reader.fieldnames:
            raise ValueError(f"{filename} must contain a text column")
        # DictReader already consumes the header.
        for row in reader:
            text = row["text"].strip()
            if not text:
                raise ValueError(f"Empty message in {filename}")
            key = " ".join(text.casefold().split())
            if key in seen:
                if seen[key] != label:
                    raise ValueError("The same message has conflicting category labels")
                continue
            seen[key] = label
            # File order defines numeric labels consistently across all splits.
            records.append({"text": text, "label": label})

    if len(records) < 10:
        raise ValueError(f"{filename} needs at least 10 unique messages")
    rng.shuffle(records)
    train_end = int(len(records) * 0.8)
    validation_end = train_end + int(len(records) * 0.1)
    split_records["train"].extend(records[:train_end])
    split_records["validation"].extend(records[train_end:validation_end])
    split_records["test"].extend(records[validation_end:])

for records in split_records.values():
    rng.shuffle(records)

dataset_features = Features({
    "text": Value("string"),
    "label": ClassLabel(names=label_names),
})
dataset = DatasetDict({
    name: Dataset.from_list(records, features=dataset_features)
    for name, records in split_records.items()
})
data_dir.mkdir(parents=True, exist_ok=True)
dataset.save_to_disk(str(data_dir / "music_service_intents_labeled"))
print(dataset)

### Load the tokenizer and labelled data

The tokenizer supplies Qwen's required conversation format; it does not train or run the model. Reloading the labelled dataset also checks that the first saved artifact can be read successfully.

The system instruction defines the six valid outputs and sends unclear or equally plausible messages to `Human_Review`.

In [ ]:
from transformers import AutoTokenizer

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
dataset = load_from_disk(str(data_dir / "music_service_intents_labeled"))
label_names = dataset["train"].features["label"].names

In [ ]:
dataset

In [ ]:
message_system = """
You route messages sent to a music service.
Classify each message as Recommendation_Request, Explanation_Request, Customer_Support, Partnership, Junk, or Human_Review.
Use Human_Review when the intent is unclear or more than one route is equally plausible.
Return only the category name, without an explanation.
""".strip()

### Format the training and validation examples

Each record becomes a three-message conversation: a system instruction, the user's message, and the correct category as the assistant response. Including the answer is appropriate here because these splits are used for learning and validation during training.

In [ ]:
def convert_to_training_messages(dataset):
 messages = [
   [
    {
        "role": "system", 
        "content": message_system
    },
    {
        "role": "user",
        "content": f"Classify: {ds['text']}"
    },
    {
        "role": "assistant",
        "content": label_names[ds["label"]]
    }
   ] for ds in dataset
 ]
 texts = [
  tokenizer.apply_chat_template(message, tokenize=False) 
    for message in messages
 ]
 return {"text": texts}

In [ ]:
ds_train = convert_to_training_messages(dataset["train"])
ds_val = convert_to_training_messages(dataset["validation"])

### Format the test examples

Test prompts include the system and user messages but omit the assistant answer. `add_generation_prompt=True` marks the point where Qwen should generate its prediction.

The numeric label remains in a separate column so evaluation code can compare the prediction with the expected category without exposing that answer in the prompt.

In [ ]:
def convert_to_prompting_messages(dataset):
  texts = []
  labels = []
  for ds in dataset:
    message = [
      {
        "role": "system", 
        "content": message_system
      },
      {
          "role": "user",
          "content": f"Classify: {ds['text']}"
      }
    ]
    chat_message = tokenizer.apply_chat_template(
      message,
      tokenize=False,
      add_generation_prompt=True,
    )
    texts.append(chat_message)
    labels.append(ds["label"])
  return {"text": texts, "label": labels}

In [ ]:
ds_test = convert_to_prompting_messages(dataset["test"])


### Assemble and save the Qwen dataset

Convert the formatted dictionaries back into Hugging Face datasets, combine the three splits, and save the result. Training and validation contain formatted `text`; test contains formatted `text` plus a separate named `ClassLabel`.

In [ ]:
ds_train = Dataset.from_dict(ds_train)
ds_val = Dataset.from_dict(ds_val)
test_features = Features({
    "text": Value("string"),
    "label": ClassLabel(names=label_names),
})
ds_test = Dataset.from_dict(ds_test, features=test_features)

In [ ]:
ds_combined = DatasetDict({
	"train": ds_train,
	"validation": ds_val,
	"test": ds_test
})

In [ ]:
ds_combined.save_to_disk(str(data_dir / "music_service_intents_qwen"))


## Results

Reload the final artifact to confirm that it was saved correctly. With the supplied twelve messages per category, the labelled dataset contains 72 unique messages:

| Split | Per intent | Total | Purpose |
| --- | ---: | ---: | --- |
| Train | 9 | 54 | Fine-tuning examples |
| Validation | 1 | 6 | Checks during training |
| Test | 2 | 12 | Unseen final evaluation |

Because the split is performed within each source file, every split contains all six intents in equal proportions.

In [ ]:
ds_llm = load_from_disk(str(data_dir / "music_service_intents_qwen"))
ds_llm

## Analysis

The preparation meets the notebook's structural goal: each intent is represented in every split, exact normalized duplicates cannot cross split boundaries, and the fixed seed makes the result reproducible. The test schema also keeps expected labels outside the model prompt.

These checks establish dataset organization, not classifier quality. A complete evaluation still needs a prompting baseline, fine-tuning, predictions on the test set, and per-category error analysis.

## Failure Cases

Exact duplicate detection does not catch paraphrases such as “Why was this suggested?” and “What caused this recommendation?”. Closely related paraphrases in different splits could make evaluation look better than real-world performance.

Some intents naturally overlap. A message about a recommended song failing to play could be an explanation request or customer support request. `Human_Review` gives the system a safe route for ambiguity, but its boundary depends on consistent labelling.

The examples are small and written for this lesson. Twelve messages per intent are enough to demonstrate the pipeline but not enough to establish dependable production behavior. Real messages may contain spelling errors, multiple requests, adversarial junk, languages not represented here, or references that require conversation history.

## Conclusion

The notebook produces two reusable artifacts. `music_service_intents_labeled` preserves cleaned messages and named labels, while `music_service_intents_qwen` stores chat-formatted training, validation, and test data. Both artifacts persist the label names wherever a label column is present.

This preparation extends the song recommendation application with a routing dataset. It does not replace the recommender or the RAG explanation workflow; it prepares a classifier that can decide which workflow should handle a message.

## From Data Preparation to Evaluation

The next stage asks a practical question: **does fine-tuning classify these messages more reliably than prompting the original model?** The prepared splits support four stages:

1. **Establish an Ollama baseline.** Load raw messages from `music_service_intents_labeled["test"]`, construct separate `system` and `user` messages, and send them to Ollama's chat API with `qwen2.5:0.5b-instruct`. Do not send the already rendered text from `music_service_intents_qwen` as a user message, because Ollama would apply its chat template again. Save each prediction rather than judging a few examples by eye.
2. **Fine-tune the same model family.** Train on the training split, use the validation split to monitor the run, and keep the test split completely unseen. Ollama is the local runtime; a separate training framework produces the model weights or adapter.
3. **Run the fine-tuned model through Ollama.** Export the result in a compatible model format, import it under a new Ollama model name, and quantize it to Q4_K_M to match the baseline build. Classify the same raw test messages with the same instructions and generation settings.
4. **Compare the two approaches.** Measure overall accuracy, per-category results, and the confusion matrix. Inspect every error, especially useful requests marked as junk, explanation requests sent away from the RAG workflow, and unclear messages that should have reached human review.

The comparison should use the unchanged test set for both models. If fine-tuning produces little improvement, the extra training and deployment complexity may not be justified. If it improves the important routing errors, the classifier can become the entry point to the application: `Recommendation_Request` goes to the recommender, `Explanation_Request` goes to the existing RAG explanation workflow, and the remaining labels go to their corresponding queues.